# Cleaning Data

- the objective of this notebook is to clean all the data collected
- firstly we clean each CSV file by:
    - removing all missing values
    - creating a JSON parser
    - changing all the models into ml ready data
- we will then create the final HexGate Archive


In [1]:
import pandas as pd
import os
import scipy as sp
import matplotlib.pyplot as plt
import numpy as np

In [2]:
PILOT_CSV = "./ToBeCleaned.csv"
df = pd.read_csv(PILOT_CSV)

df.head(10)

,GAME_ID,BLUE_WIN,BLUE_KILLS,RED_KILLS,BLUE_DEATHS,RED_DEATHS,RED_ASSISTS,BLUE_ASSISTS,BLUE_WARDS_PLACED,RED_WARDS_PLACED,...,RED_TEAM_ADC_SUMMONER_01,RED_TEAM_ADC_SUMMONER_02,RED_TEAM_SUP_SUMMONER_01,RED_TEAM_SUP_SUMMONER_02,BLUE_TEAM_RIFT_HERALDS,BLUE_TEAM_VOID_GRUBS,BLUE_TEAM_INHIBITORS,RED_TEAM_RIFT_HERALDS,RED_TEAM_VOID_GRUBS,RED_TEAM_INHIBITORS
0,https://www.op.gg/summoners/euw/Walid%20George...,1.0,36,23,23,36,37,62,80,73,...,Flash,Barrier,Flash,Ignite,1,5,1,0,1,0
1,NaN,NaN,-1,-1,-1,-1,-1,-1,-1,-1,...,NaN,NaN,NaN,NaN,-1,-1,-1,-1,-1,-1
2,https://www.op.gg/summoners/euw/Walid%20George...,1.0,34,36,36,34,62,62,101,71,...,Barrier,Flash,Flash,Ignite,0,1,1,1,5,0
3,https://www.op.gg/summoners/euw/Walid%20George...,0.0,25,36,36,25,62,40,90,67,...,Barrier,Flash,Ignite,Flash,0,6,0,1,0,2
4,https://www.op.gg/summoners/euw/Walid%20George...,1.0,36,26,26,37,59,53,71,78,...,Flash,Teleport,Flash,Heal,1,5,2,0,1,0
5,NaN,NaN,-1,-1,-1,-1,-1,-1,-1,-1,...,NaN,NaN,NaN,NaN,-1,-1,-1,-1,-1,-1
6,https://www.op.gg/summoners/euw/Walid%20George...,1.0,23,11,11,23,26,41,36,33,...,Flash,Teleport,Flash,Heal,1,6,1,0,0,0


In [3]:
empty_cells = df[df.isna().any(axis=1)]

empty_cells.head(5)

,GAME_ID,BLUE_WIN,BLUE_KILLS,RED_KILLS,BLUE_DEATHS,RED_DEATHS,RED_ASSISTS,BLUE_ASSISTS,BLUE_WARDS_PLACED,RED_WARDS_PLACED,...,RED_TEAM_ADC_SUMMONER_01,RED_TEAM_ADC_SUMMONER_02,RED_TEAM_SUP_SUMMONER_01,RED_TEAM_SUP_SUMMONER_02,BLUE_TEAM_RIFT_HERALDS,BLUE_TEAM_VOID_GRUBS,BLUE_TEAM_INHIBITORS,RED_TEAM_RIFT_HERALDS,RED_TEAM_VOID_GRUBS,RED_TEAM_INHIBITORS
1,NaN,NaN,-1,-1,-1,-1,-1,-1,-1,-1,...,NaN,NaN,NaN,NaN,-1,-1,-1,-1,-1,-1
5,NaN,NaN,-1,-1,-1,-1,-1,-1,-1,-1,...,NaN,NaN,NaN,NaN,-1,-1,-1,-1,-1,-1


In [4]:
clean_cells = df.dropna()

clean_cells.head(10)

,GAME_ID,BLUE_WIN,BLUE_KILLS,RED_KILLS,BLUE_DEATHS,RED_DEATHS,RED_ASSISTS,BLUE_ASSISTS,BLUE_WARDS_PLACED,RED_WARDS_PLACED,...,RED_TEAM_ADC_SUMMONER_01,RED_TEAM_ADC_SUMMONER_02,RED_TEAM_SUP_SUMMONER_01,RED_TEAM_SUP_SUMMONER_02,BLUE_TEAM_RIFT_HERALDS,BLUE_TEAM_VOID_GRUBS,BLUE_TEAM_INHIBITORS,RED_TEAM_RIFT_HERALDS,RED_TEAM_VOID_GRUBS,RED_TEAM_INHIBITORS
0,https://www.op.gg/summoners/euw/Walid%20George...,1.0,36,23,23,36,37,62,80,73,...,Flash,Barrier,Flash,Ignite,1,5,1,0,1,0
2,https://www.op.gg/summoners/euw/Walid%20George...,1.0,34,36,36,34,62,62,101,71,...,Barrier,Flash,Flash,Ignite,0,1,1,1,5,0
3,https://www.op.gg/summoners/euw/Walid%20George...,0.0,25,36,36,25,62,40,90,67,...,Barrier,Flash,Ignite,Flash,0,6,0,1,0,2
4,https://www.op.gg/summoners/euw/Walid%20George...,1.0,36,26,26,37,59,53,71,78,...,Flash,Teleport,Flash,Heal,1,5,2,0,1,0
6,https://www.op.gg/summoners/euw/Walid%20George...,1.0,23,11,11,23,26,41,36,33,...,Flash,Teleport,Flash,Heal,1,6,1,0,0,0


In [5]:
import json
import re
import os
import lol_context as lc
import pandas as pd

df = lc.Champion

def get_cc_stats(champion_list):
    data_dragon = []

    for champion in champion_list:
        result = []
        with open("./en_US/champion/"+champion, mode="r", encoding="utf-8") as read_file:
            data_phoenix = json.load(read_file)

            # Collect All the Crowd Control stats
            total_cc_score = 0
            crowd_control = re.findall(r"<status>(.*?)</status>", str(data_phoenix))

            for cc in crowd_control:
                if "Stun" in cc or "Root" in cc:
                    total_cc_score +=3
                elif "Knock" in cc:
                    total_cc_score +=2
                else:
                    total_cc_score +=1

            result.append(total_cc_score)

            # Collect the champion key
            champ_id = data_phoenix['data'][champion.replace(".json", "")]['key']
            result.append(champ_id)

            # Collect the champion name
            champ_name = champion.replace(".json", "").lower()
            result.append(champ_name)

            # Collect champion stats
            # They are [attack, defense, magic, difficulty]
            champ_stats = list(data_phoenix['data'][champion.replace(".json", "")]['info'].values())
            result.append(champ_stats)

            # Collect champion tags
            tag_list = ['Fighter', 'Marksman', 'Tank', 'Assassin', 'Support', 'Mage']

            sub_list = data_phoenix['data'][champion.replace(".json", "")]['tags']
            result.append([1 if tag in sub_list else 0 for tag in tag_list])

        data_dragon.append(result)

    return data_dragon

x = get_cc_stats(os.listdir("./en_US/champion/"))


column_names = ["CC_SCORE", "CHAMP_ID", "CHAMPION_NAME", "CHAMPION_INFO", "CHAMPION_TAGS"]
df = pd.DataFrame(x, columns=column_names)

df.head(5)

,CC_SCORE,CHAMP_ID,CHAMPION_NAME,CHAMPION_INFO,CHAMPION_TAGS
0,5,266,aatrox,"[8, 4, 3, 4]","[1, 0, 0, 0, 0, 0]"
1,1,103,ahri,"[3, 4, 8, 5]","[0, 0, 0, 1, 0, 1]"
2,1,84,akali,"[5, 3, 8, 7]","[0, 0, 0, 1, 0, 0]"
3,0,166,akshan,"[0, 0, 0, 0]","[0, 1, 0, 1, 0, 0]"
4,9,12,alistar,"[6, 9, 5, 7]","[0, 0, 1, 0, 1, 0]"


In [6]:
blue_team = clean_cells[[
    'BLUE_TEAM_TOP',
    'BLUE_TEAM_JNG',
    'BLUE_TEAM_MID',
    'BLUE_TEAM_ADC',
    'BLUE_TEAM_SUP'
]]

red_team = clean_cells[[
    'RED_TEAM_TOP',
    'RED_TEAM_JNG',
    'RED_TEAM_MID',
    'RED_TEAM_ADC',
    'RED_TEAM_SUP'
]]

print(red_team)

  RED_TEAM_TOP RED_TEAM_JNG RED_TEAM_MID RED_TEAM_ADC RED_TEAM_SUP
0     Cho'Gath      Kindred       Syndra      Caitlyn        Poppy
2       Gragas       Graves     Kassadin         Ashe     Nautilus
3        Sylas         Zyra       Akshan         Jinx      Alistar
4         Gnar     Bel'Veth       Syndra    Seraphine        Senna
6         Yone      Skarner        Galio        Swain         Nami


In [7]:
sln = []

for x in red_team.iterrows():
    team_list = [x[1][0], x[1][1], x[1][2], x[1][3], x[1][4]]
    result = []

    for name in team_list:
        result.append(str(name).replace("\'", "").lower())

    sln.append(result)

new_sln = pd.DataFrame(sln)

champion_names.reverse()

shurima_dict = []

for x in df.iterrows():
    shurima_dict.append((x[1][2], int(x[1][1])))

sundisk = dict(shurima_dict)

new_red_team = new_sln.replace(sundisk)

print(new_red_team)


with open("champion_dictionary.json", mode="w") as file:
    json.dump(sundisk, file, indent=4)

C:\Users\tinay\AppData\Local\Temp\ipykernel_1396\725473785.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  team_list = [x[1][0], x[1][1], x[1][2], x[1][3], x[1][4]]


NameError: name 'champion_names' is not defined